trial on CU_2538; gitignore this file

We have the re-coregistered image (cerebellum-only coregistration) for the cerebellum.

Now:

- assign its affine to the T1 anat for that week (W4)

- assign its affine to the wm probability map for that week (W4)

- run regression on this: copy over the avg_vol fcn, change the inputs to take the updated_affine images, and make the directory this one. Also need to copy over the wm files for W0, W4 into this directory - since we only run the regression on white matter.

In [1]:
# Imports

# model libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# image reading libraries
import nibabel as nib
from nilearn import plotting as npl
from nilearn import masking # for masking within-brain voxels

import nitools as nt

from pathlib import Path
import os

In [2]:
# directories
base_dir = '/cifs/diedrichsen/data/smarts_cerebellum'
anat_dir = '/cifs/diedrichsen/data/smarts_cerebellum/anatomicals'
p_df = pd.read_csv('/cifs/diedrichsen/data/smarts_cerebellum/participants_anat.tsv', sep = '\t')

In [3]:
df = p_df[p_df.subj_id == 'CU_2538']

In [4]:
subj_id = 'CU_2538'
refT1 = 'W0'
week = 'W4'

In [5]:
df

,SN,ID,Centre,Week,week,RefT1,numrun,nslices,Hand,LesionSide,...,surfmvpa,include1,lesiondef,behavior_missing,behavior_blocks,has_mvc,DTImap_missing,CentreNo,machine,subj_id
5,6,2538,CU,W0,0,W0,8,35,b,right,...,1,1,1,0,8,1,0,1,naveed,CU_2538
6,7,2538,CU,W4,4,W0,8,35,b,right,...,1,1,1,0,8,1,0,1,naveed,CU_2538


In [6]:
cerebel_ref_path = f'{anat_dir}/{subj_id}/{refT1}/{subj_id}_{refT1}_cerebellum_only_trial.nii'
cerebel_week_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_cerebellum_only_trial.nii'

In [7]:
cerebel_ref_img = nib.load(cerebel_ref_path)
cerebel_week_img = nib.load(cerebel_week_path)

In [8]:
cerebel_week_img.affine

array([[-5.04541397e-03,  4.84132171e-02,  1.19858980e+00,
        -1.04071968e+02],
       [-9.98276830e-01, -5.86394072e-02, -2.64048576e-03,
         1.24560898e+02],
       [-5.84633350e-02,  9.97104585e-01, -5.83513975e-02,
        -1.13767464e+02],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00]])

In [9]:
t1_week_path = f'{anat_dir}/{subj_id}/{week}/{subj_id}_{week}_T1.nii'
c2_week_path = f'{anat_dir}/{subj_id}/{week}/c2{subj_id}_{week}_T1.nii'

In [10]:
t1_week_img = nib.load(t1_week_path)
c2_week_img = nib.load(c2_week_path)

set affines of t1 and c2 imgs (and sform), and save as _updated_affine.nii.gz

In [11]:
t1_week_img.affine[:] = cerebel_week_img.affine
c2_week_img.affine[:] = cerebel_week_img.affine

Try without updating q-form, s-form

In [12]:
nib.save(t1_week_img, f'{subj_id}_{week}_T1_updated_affine.nii.gz')
nib.save(c2_week_img, f'c2{subj_id}_{week}_T1_updated_affine.nii.gz')

# Now onto the regression

Will just copy the function here and edit it for testing instead of editing in its actual file.

In [13]:
# add input: type of file (e.g. native, tissue_resliced, etc.)
def avg_vol(subj_id, 
            reference_img, # reference anatomical
            #week_path, # path to week image
            results_path,
            image_suffix,
            tissue=None):
    """
    Inputs:
    anat dir: participants file OR [(subj, week) and call it inside a loop].
    Reference img (inside the Jupyter notebook loop for reading off the info file)
    #week_path (path for each week's image), results_path (store results)
    results path: directory to store results
    image suffix: suffix with which to save the slope and intercept images
        suggested: <image_type>_<space> where 'image_type' is "anat", "wm", "gm", etc; 'space' is native or template (<template_name>)

    Everything is done in the reference image. So this function will (...) (resample voxels in other weeks so that they are aligned with the reference, and perform multiple linear regression)

    Returns B_hat coefficient matrix (for more flexibility in other possible operations)
    
    """

    # file prefix encoding (based on SPM segmentation notation)
    tissue_dict = {
        'gm': 'c1',
        'wm': 'c2',
        'csf': 'c2'
    }

    img0 = nib.load(reference_img)

    # later fix: option to reduce to only wtihin-brain voxels
    
    # transform into world coordinates
    i, j, k = np.indices(img0.shape) # matrix indices for premult by affine
    x,y,z = nt.affine_transform(i, j, k, img0.affine)

    # all possible weeks.
    weeks = np.array([0,4,12,24,52]) # read from file, use file reading function maybe
    
    
    # THIS PART SHOULD BE DONE IN TUTORIAL, NOT IN FUNCTION. or with a helper function.


    #____________________________________
    # find the number of measurement weeks that exist
    p_weeks = []
    for week in weeks:
        #week_path = f'{anat_dir}/{subj_id}/W{week}/wm_results/{subj_id}_W{week}_T1_wm_vol.nii'
        #week_path = week_path
        #week_path = f'{anat_dir}/{subj_id}/W{week}/c2{subj_id}_W{week}_T1.nii' # fix

        if not tissue==None:
            week_path = f'{tissue_dict[tissue]}{subj_id}_W{week}_T1_updated_affine.nii'
        else:
            print("no tissue specified")


        # skip over missed measurement weeks.
        if not os.path.exists(week_path):
            continue

        p_weeks.append(week)
    
    if len(p_weeks) == 1: # only one measurement week available
        return None # exit function (skip subject)    
    #__________________________


    Y = np.zeros((len(p_weeks), np.prod(img0.shape))) # initialize Y (shape = (k by p)) array, where k = number of weeks available

    for week in weeks: # resample ALL weeks, including the reference week
        #week_path = f'{anat_dir}/{subj_id}/W{week}/wm_results/{subj_id}_W{week}_T1_wm_vol.nii'
        #week_path = week_path
        
        if not tissue==None:
            week_path = f'{tissue_dict[tissue]}{subj_id}_W{week}_T1_updated_affine.nii'
            #print(f'using {tissue}')
        else:
            print("no tissue specified 2")

        # skip over missed measurement weeks.
        if not os.path.exists(week_path):
            continue

        week_img = nib.load(week_path)
        print(f'on week {week} for {subj_id}')

        week_dict = {
            '0': 0,
            '4': 1,
            '12': 2,
            '24': 3,
            '52': 4
        }

        print(week_path)
        
        # resample each week's image so that voxels are exactly on top of reference week voxels; add to response matrix as row vector
        Y[week_dict[str(week)]:,] = nt.sample_image(week_img, # response matrix
                                xm=x, ym = y, zm = z, # world coordinates
                                interpolation = 1 # using trilinear resampling
                                ).flatten() # need to put each week as a row
        
        # now we have Y as a k by p matrix, where k is the number of weeks.  

    # design matrix
    num_weeks = len(p_weeks)
    X = [np.ones(shape = (num_weeks)), p_weeks]
    X = np.array(X)
    X = X.T

    # estimator (coefficients matrix)
    B_hat = np.linalg.pinv(X) @ Y
    # where B_hat = [B_0 B_1].T

    #_________________
    # save image with voxel coordinates
    slope = np.zeros(img0.shape) # tensor with shape of reference img
    intercept = np.zeros(img0.shape)
    # need i, j, k as vectors (they're tensors right now)

    iv = i.flatten()
    jv = j.flatten()
    kv = k.flatten()

    slope[iv, jv, kv] = B_hat[1,:] # write the slope into the vectors i, j, k for coordinates
    intercept[iv, jv, kv] = B_hat[0,:]

    intercept_img = nib.Nifti1Image(intercept, img0.affine)
    slope_img = nib.Nifti1Image(slope, img0.affine)

    nib.save(intercept_img, f'{results_path}/{subj_id}_T1_intercept_{image_suffix}.nii.gz') # specify file name
    nib.save(slope_img, f'{results_path}/{subj_id}_T1_slope_{image_suffix}.nii.gz')

    

    return B_hat


In [14]:
avg_vol?

Signature: avg_vol(subj_id, reference_img, results_path, image_suffix, tissue=None)
Docstring:
Inputs:
anat dir: participants file OR [(subj, week) and call it inside a loop].
Reference img (inside the Jupyter notebook loop for reading off the info file)
#week_path (path for each week's image), results_path (store results)
results path: directory to store results
image suffix: suffix with which to save the slope and intercept images
    suggested: <image_type>_<space> where 'image_type' is "anat", "wm", "gm", etc; 'space' is native or template (<template_name>)

Everything is done in the reference image. So this function will (...) (resample voxels in other weeks so that they are aligned with the reference, and perform multiple linear regression)

Returns B_hat coefficient matrix (for more flexibility in other possible operations)
File:      /localscratch/tmp/ipykernel_78406/2221445418.py
Type:      function

In [15]:
ref_img = f'c2{subj_id}_{week}_T1_updated_affine.nii'

In [16]:
betas = avg_vol(
    subj_id = subj_id,
    reference_img = ref_img,
    results_path = '/home/UWO/mporwal2/Documents/GitHub/smarts_cerebellum/cerebellar_alignment',
    image_suffix = 'native_wm_updated_affine',
    tissue = 'wm'
)

on week 0 for CU_2538
c2CU_2538_W0_T1_updated_affine.nii
on week 4 for CU_2538
c2CU_2538_W4_T1_updated_affine.nii


In [17]:
betas[1].max()

np.float64(0.2500000147847458)